# 股票、现金与对冲误差

每次调仓显式更新现金，用多路径分布评价离散误差。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

先核对两次持仓的现金流。

In [ ]:
price=10.;S0=100.;delta0=.6;cost=.001
cash=price-S0*delta0-cost*S0*abs(delta0)
S1=110.;delta1=.8;trade=delta1-delta0;cash-=S1*trade+cost*S1*abs(trade)
ST=105.;payoff=max(ST-100,0);error=cash+delta1*ST-payoff
print('cash, terminal wealth, payoff, error:',cash,cash+delta1*ST,payoff,error)

使用 GBM 路径与模型 Delta。到期只估值，不额外调仓。

In [ ]:
S0=100.;K=100.;r=.02;vol=.2;T=1.;N=2000;steps=24;dt=T/steps
def delta(s,tau):return stats.norm.cdf((np.log(s/K)+(r+.5*vol**2)*tau)/(vol*np.sqrt(tau)))
d1=(np.log(S0/K)+(r+.5*vol**2)*T)/(vol*np.sqrt(T));price=S0*stats.norm.cdf(d1)-K*np.exp(-r*T)*stats.norm.cdf(d1-vol*np.sqrt(T))
Z=rng.normal(size=(N,steps));paths=S0*np.exp(np.cumsum((r-.5*vol**2)*dt+vol*np.sqrt(dt)*Z,axis=1))
results=[]
for c in [0.,.001]:
    holding=np.full(N,delta(S0,T));cash=np.full(N,price)-holding*S0-c*S0*abs(holding)
    for k in range(steps):
        spot=paths[:,k];cash*=np.exp(r*dt)
        if k<steps-1:
            new=delta(spot,T-(k+1)*dt);trade=new-holding;cash-=spot*trade+c*spot*abs(trade);holding=new
    errors=cash+holding*paths[:,-1]-np.maximum(paths[:,-1]-K,0);results.append(errors)
    print(c,'bias, RMSE, quantiles:',errors.mean(),np.sqrt(np.mean(errors**2)),np.quantile(errors,[.05,.5,.95]))
plt.hist(results[0],bins=50,alpha=.5,label='no cost');plt.hist(results[1],bins=50,alpha=.5,label='with cost');plt.legend();plt.show()

## 自己试一试

增加 steps 再运行；为什么不能只根据误差宽度选频率？

## 反馈

更频繁调仓通常降低离散误差但增加交易成本，还需比较均值、尾部、费用与资金占用。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。